In [ ]:
from pathlib import Path
from config import LOCAL_VOLUME, PROJECT_ROOT

import modal

app = modal.App("LLM-pretraining")

container = (
    modal.Image.debian_slim(python_version="3.12")
    .uv_sync(uv_project_dir=PROJECT_ROOT)
    .add_local_python_source("transformer")
)

volume = modal.Volume.from_name("LLM-pretraining", create_if_missing=True)

In [ ]:
@app.cls(image=container, volumes={"/storage": volume})
class ETL:
    """Data ETL for the pretraining pipeline. Three independent, re-runnable
    stages, each committed to the Volume so a later stage (in a fresh container)
    sees prior output. Training is a separate pipeline -- not part of ETL.

    Layout (every path relative to VOLUME):
        data/{dataset_name}/{split}.txt                       raw text  (download)
        tokenizers/{tokenizer_uid}/...                        tokenizer (train_tokenizer)
        data/{dataset_name}/bin/{tokenizer_uid}/{split}.bin   encoded   (tokenize)

    Granularity is the point: download a dataset once, train N tokenizers against
    any raw text, and tokenize the same dataset under whichever tokenizer_uid you
    like -- the per-tokenizer bin/ namespace lets those outputs coexist.

    `remote` drives everything location-specific: VOLUME (LOCAL_VOLUME vs the
    mount) and the commit/reload calls (Volume-API ops that only mean anything
    remotely -- locally we read/write disk directly).

    NOTE these are properties, not class attributes. A class attribute is computed
    once, when the class is DEFINED (on the laptop, where is_local() is True); for
    a notebook-defined class that frozen value is shipped to the container as-is,
    so it'd wrongly read `remote=False` remotely and never commit -> nothing lands
    on the Volume. A property re-evaluates is_local() at access time, on whichever
    side is actually running.
    """

    @property
    def remote(self) -> bool:
        return not modal.is_local()

    @property
    def VOLUME(self) -> Path:
        return Path("/storage") if self.remote else LOCAL_VOLUME

    @modal.method()
    def download(self, dataset_name: str, data_sources: dict[str, list[str]]):
        """Download + concat each split's URLs into data/{dataset_name}/{split}.txt.
        Independent of any tokenizer -- run once per dataset. Overwrites."""
        from transformer.util import download_and_concat

        for split, urls in data_sources.items():
            download_and_concat(
                urls,
                f"data/{dataset_name}/{split}.txt",
                self.VOLUME,
                separator="\n<|endoftext|>\n",
            )
        if self.remote:
            volume.commit()  # persist so later stages / containers see the text

    @modal.method()
    def train_tokenizer(
        self,
        tokenizer_uid: str,
        raw_text_path: str,
        vocab_size: int,
        special_tokens: list[str],
    ):
        """Train a BPE tokenizer on raw_text_path (volume-relative, e.g.
        "data/gutenberg/tokens.txt") and write it under tokenizers/{tokenizer_uid}/.
        Decoupled from any dataset -- reuse a uid across datasets, or train many
        uids against the same text. Overwrites an existing uid."""
        from transformer.util import prepare_tokenizer

        if self.remote:
            volume.reload()  # see raw text a prior download stage committed
        prepare_tokenizer(tokenizer_uid, vocab_size, special_tokens, raw_text_path, self.VOLUME)
        if self.remote:
            volume.commit()

    @modal.method()
    def tokenize(
        self,
        dataset_name: str,
        tokenizer_uid: str,
        splits: tuple[str, ...] = ("train", "valid"),
    ):
        """Encode each split of a dataset with a tokenizer into
        data/{dataset_name}/bin/{tokenizer_uid}/{split}.bin. Per-tokenizer
        namespacing lets multiple tokenizers coexist on the same dataset."""
        from transformer.tokenizer import Tokenizer
        from transformer.util import textfile_to_tokens_as_binary

        if self.remote:
            volume.reload()  # see txt + tokenizer prior stages committed
        tokenizer = Tokenizer.from_files(tokenizer_uid, self.VOLUME)

        bin_dir = self.VOLUME / "data" / dataset_name / "bin" / tokenizer_uid
        # Create the dir BEFORE writing into it -- on the Volume, a write with a
        # missing parent doesn't fail fast, it hangs.
        bin_dir.mkdir(parents=True, exist_ok=True)
        for split in splits:
            textfile_to_tokens_as_binary(
                f"data/{dataset_name}/{split}.txt",
                f"data/{dataset_name}/bin/{tokenizer_uid}/{split}.bin",
                tokenizer,
                self.VOLUME,
            )
        if self.remote:
            volume.commit()

In [ ]:
# --- Example driver: each stage is a separate call, run only what you need ---
dataset_name = "gutenberg"
data_sources = {
    "tokens": ["https://gutenberg.org/cache/epub/1184/pg1184.txt"],  # raw text for tokenizer training
    "train": ["https://gutenberg.org/cache/epub/1184/pg1184.txt"],
    "valid": ["https://gutenberg.org/cache/epub/1513/pg1513.txt"],
}

tokenizer_uid = "gutenberg-bpe-1000"
special_tokens = ["<|endoftext|>", "<|begin|>", "<|end|>"]
vocab_size = 1000

# Flip this one variable: "remote" runs every stage in a Modal container against
# the mounted Volume; "local" runs them in this process against LOCAL_VOLUME.
mode = "remote"  # "remote" | "local"


def run(method, *args):
    return getattr(method, mode)(*args)


with modal.enable_output():
    with app.run():
        etl = ETL()
        # 1. download once per dataset
        run(etl.download, dataset_name, data_sources)
        # 2. train one (or more) tokenizers against any raw text
        run(etl.train_tokenizer, tokenizer_uid, f"data/{dataset_name}/tokens.txt", vocab_size, special_tokens)
        # 3. tokenize the dataset's train/valid splits under a chosen tokenizer
        run(etl.tokenize, dataset_name, tokenizer_uid)